# Nested seasonality: one fold function, applied twice

Demonstrates the FLAIR `level / shape` idea recursively. The data has a
**weekly** rhythm and a **monthly** rhythm on top of a slow trend. We use ONE
reusable function `deseasonalize(series, period)` and apply it **twice** -- once
on the daily series (weekly), once on the resulting weekly series (monthly) --
to peel both seasonalities, forecast only the smooth trend, then reassemble.

You will see, side by side, that **one** fold (weekly only) leaves the monthly
swing unmodeled (large error), while **two** folds flatten the series to a line
and forecast it almost perfectly.


## 1. The reusable fold function

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
np.set_printoptions(precision=3, suppress=True)

def deseasonalize(series, period):
    """Fold `series` by `period`, extract a mean-1 seasonal factor, and return
    the deseasonalized COARSE series (one value per cycle).

    Returns
    -------
    factor : (period,)        mean-1 multiplicative seasonal pattern
    coarse : (n_cycles,)      series with that seasonality divided out, one per cycle
    M      : (n_cycles, period) the folded matrix (for inspection)
    """
    n = len(series) // period
    M = series[-n * period:].reshape(n, period)        # (cycles, period)
    mean = M.mean(axis=1, keepdims=True)
    mean = np.where(mean == 0, 1.0, mean)
    factor = (M / mean).mean(axis=0)                   # average phase pattern
    factor = factor / factor.mean()                    # normalize to mean 1
    coarse = (M / factor[None, :]).mean(axis=1)        # deseasonalized, 1 / cycle
    return factor, coarse, M


## 2. Build a daily series with weekly + monthly seasonality + trend

In [ ]:
# truth:  daily = trend[month] * monthFactor[week-of-month] * weekFactor[day-of-week]
DOW   = np.array([1.0, 1.1, 1.2, 1.0, 1.3, 0.6, 0.5])   # Mon..Sun
DOW   = DOW / DOW.mean()                                # mean 1
WOM   = np.array([1.2, 1.0, 0.9, 0.9])                  # week-of-month, mean 1
N_TRAIN_MONTHS = 6
H_MONTHS       = 1                                      # forecast 1 month (28 days)

def month_trend(m):       # smooth growing level, one number per month
    return 1000 * (1 + 0.05 * m)

def make_daily(n_months, noise=0.0, seed=0):
    rng = np.random.default_rng(seed)
    out = []
    for m in range(n_months):
        for w in range(4):
            week = month_trend(m) * WOM[w] * DOW
            if noise:
                week = week * (1 + rng.normal(0, noise, 7))
            out.append(week)
    return np.concatenate(out)

full   = make_daily(N_TRAIN_MONTHS + H_MONTHS, noise=0.0)
train  = full[:N_TRAIN_MONTHS * 28]
future = full[N_TRAIN_MONTHS * 28:]                     # the 28 days we will predict

plt.figure(figsize=(13, 4))
plt.plot(np.arange(len(train)), train, label="history (daily)")
plt.plot(np.arange(len(train), len(full)), future, color="black", lw=2, label="true future")
plt.axvline(len(train), color="gray", ls="--")
plt.title("Daily series: weekly wiggle inside a monthly wiggle inside a trend")
plt.legend(); plt.show()


## 3. Layer 1 -- fold by the WEEK (period = 7)

Extract the day-of-week pattern and collapse to one number per week. The
resulting weekly series is still wiggly -- it carries the monthly rhythm.

In [ ]:
dow_factor, weekly, M7 = deseasonalize(train, period=7)
print("day-of-week factor (mean 1):", dow_factor)
print("weekly coarse series       :", weekly.round(1))

fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
ax[0].bar(range(7), dow_factor); ax[0].axhline(1, color="k", lw=.7)
ax[0].set_title("Layer 1 factor: day-of-week (mean 1)")
ax[0].set_xticks(range(7)); ax[0].set_xticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
ax[1].plot(weekly, "-o"); ax[1].set_title("weekly coarse series (still has the MONTHLY wiggle)")
ax[1].set_xlabel("week index"); plt.tight_layout(); plt.show()


## 4. Layer 2 -- the SAME function on the weekly series (period = 4)

Fold the weekly series by 4 (a month). Extract the week-of-month pattern and
collapse to one number per month -- now it's a smooth trend.

In [ ]:
wom_factor, monthly, M4 = deseasonalize(weekly, period=4)
print("week-of-month factor (mean 1):", wom_factor)
print("monthly trend (smooth!)      :", monthly.round(1))

fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
ax[0].bar(range(4), wom_factor); ax[0].axhline(1, color="k", lw=.7)
ax[0].set_title("Layer 2 factor: week-of-month (mean 1)")
ax[0].set_xticks(range(4)); ax[0].set_xticklabels(["wk1","wk2","wk3","wk4"])
ax[1].plot(weekly/ wom_factor[np.arange(len(weekly)) % 4], "-o", label="weekly / wom  (flattened)")
ax[1].plot(np.repeat(monthly, 4), "--", label="monthly trend (repeated)")
ax[1].set_title("after removing BOTH seasonalities -> a line"); ax[1].legend()
ax[1].set_xlabel("week index"); plt.tight_layout(); plt.show()


## 5. Forecast: extrapolate the trend, then multiply the two factors back

`forecast_day = trend_hat * wom_factor[week-of-month] * dow_factor[day-of-week]`

In [ ]:
H_DAYS = H_MONTHS * 28

# extrapolate the smooth monthly trend one step (linear fit)
mm = np.arange(len(monthly))
slope, intercept = np.polyfit(mm, monthly, 1)
trend_next = slope * len(monthly) + intercept
print("forecast trend for next month:", round(trend_next, 1))

# reassemble the 28-day forecast
two_level = np.concatenate([trend_next * wom_factor[w] * dow_factor for w in range(4)])

# --- baseline: ONE fold only (weekly), no monthly factor ---
ww = np.arange(len(weekly))
s1, i1 = np.polyfit(ww, weekly, 1)
one_level = np.concatenate([(s1 * (len(weekly) + w) + i1) * dow_factor for w in range(4)])

def mape(p, t): return float(np.mean(np.abs(p - t) / t) * 100)
print(f"MAPE one fold  (weekly only)      : {mape(one_level, future):.2f}%")
print(f"MAPE two folds (weekly + monthly) : {mape(two_level, future):.2f}%")

t = np.arange(H_DAYS)
plt.figure(figsize=(13, 4))
plt.plot(t, future, color="black", lw=2, label="true future")
plt.plot(t, one_level, "--", label=f"1 fold (weekly only)  MAPE {mape(one_level,future):.1f}%")
plt.plot(t, two_level, "--", label=f"2 folds (weekly+monthly)  MAPE {mape(two_level,future):.1f}%")
plt.title("Forecast: one fold leaves the monthly swing; two folds capture it")
plt.legend(); plt.show()


## 6. Takeaway

- Layer 1 (weekly fold) removes the day-of-week pattern -> a weekly series that
  still carries the monthly rhythm.
- Layer 2 is the **same function** applied to that weekly series -> removes the
  week-of-month pattern -> a smooth monthly trend.
- You only forecast the smooth trend, then multiply the two factors back.
- One fold leaves the second seasonality unmodeled (big error); two folds make
  it nearly exact. Adding a third period (e.g. yearly) is just another call.
